# ContinuumFusion - arbitrary-scale implicit representation

Proposal 3 of four. Proposals 1 and 2 both produce a fixed output grid and are
welded to one scale factor; changing it means retraining.

ContinuumFusion never represents the image as a grid. It learns a continuous
function `f(x, y, lambda) -> radiance`, conditioned on local features from the
observations, and **samples** it wherever an output pixel is wanted. The scale
factor becomes a query parameter rather than an architectural constant.

* Latent features live on the **LR grid**, because that is where the spectra
  were actually measured. Spatial detail is read at query time from the
  full-resolution MSI.
* The band index is a **coordinate**, entering through a learned embedding, so
  one MLP serves every band and the representation is continuous along
  wavelength as well as space.
* A local ensemble over the four neighbouring latent cells removes the blocking
  that decoding from the single nearest cell produces.

**Why this gap matters:** the benchmark in `existing/` is unusable precisely
because its ten methods ran at x4, x8, x16 and x32. That is not sloppy
bookkeeping, it is a property of grid-based architectures. A model that handles
any factor from one set of weights makes the comparison well-posed.

---

**Shared protocol.** This notebook imports the same `hsifusion` library as the
other three proposals: same degradation, same scale factor, same metric code
with a fixed `data_range=1.0`, same classical baselines, same scenes. A
difference between proposals is therefore attributable to the architecture and
not to one of them quietly evaluating differently - which is exactly how the
ten published baselines in `existing/` became incomparable.

**Hardware.** Choose **GPU T4 x2**. PyTorch >= 2.6 ships no kernels for the
P100's `sm_60`, so a P100 fails on the first CUDA op whatever the code does.
The next cell checks this and tells you if the accelerator is wrong.

## 1. Environment

In [ ]:
import os, sys, json, time, math, warnings
warnings.filterwarnings('ignore', category=UserWarning)
import torch, numpy as np

print('python  ', sys.version.split()[0])
print('torch   ', torch.__version__)
GPU_OK = False
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    arch = f'sm_{p.major}{p.minor}'
    built = list(torch.cuda.get_arch_list())
    print('gpu     ', p.name, f'{p.total_memory / 2**30:.1f} GB', arch)
    print('built   ', ' '.join(built))
    GPU_OK = arch in built
    if not GPU_OK:
        print('\n' + '*' * 74)
        print(f'INCOMPATIBLE GPU: this torch build has no kernels for {arch}.')
        print('  FIX: Settings -> Accelerator -> "GPU T4 x2", then re-run all.')
        print('*' * 74)
else:
    print('no GPU - this will be very slow on CPU')

WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
os.chdir(WORK)
print('workdir ', os.getcwd())

## 2. Shared library (`hsifusion`)

Identical in all four proposal notebooks: protocol, data pipeline, metrics,
degradation model, classical baselines and the model-agnostic engine.

In [ ]:
import os; os.makedirs('hsifusion', exist_ok=True)

In [ ]:
%%writefile hsifusion/io_utils.py
"""Filesystem discovery and .mat reading.

This module deliberately has no dependency on the rest of the package so that
dataset discovery can never create an import cycle.
"""

from __future__ import annotations

import glob
import os
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np

try:
    import scipy.io as sio
except ImportError:  # pragma: no cover
    sio = None

SPLIT_NAMES = ("Train", "train", "TRAIN")
TEST_NAMES = ("Test", "test", "TEST", "Val", "val")


# --------------------------------------------------------------------------- IO
def load_mat(path: str) -> np.ndarray:
    """Return the first real array stored in a MATLAB file."""
    if sio is None:
        raise RuntimeError("scipy is required to read .mat files")
    mat = sio.loadmat(path)
    for k, v in mat.items():
        if not k.startswith("__") and isinstance(v, np.ndarray) and v.ndim >= 2:
            return np.asarray(v)
    raise ValueError(f"no array in {path}")


def to_chw01(arr: np.ndarray, channels: int) -> np.ndarray:
    """Normalise an array to channel-first float32 in [0, 1]."""
    if channels is None:
        raise ValueError("channel count is unresolved - call Config.resolve() first")
    a = np.squeeze(np.asarray(arr)).astype(np.float32)
    if a.ndim != 3:
        raise ValueError(f"expected 3D array, got {a.shape}")
    if a.shape[0] == channels:
        pass
    elif a.shape[-1] == channels:
        a = np.transpose(a, (2, 0, 1))
    else:
        raise ValueError(f"cannot find {channels} channels in {a.shape}")
    mx = float(a.max())
    if mx > 1.0:
        a = a / mx
    return np.clip(a, 0.0, 1.0)


# ------------------------------------------------------------------- discovery
def _looks_like_dataset(path: str) -> bool:
    """A dataset root is any directory holding <split>/HSI."""
    for split in SPLIT_NAMES + TEST_NAMES:
        d = os.path.join(path, split)
        if os.path.isdir(d) and any(
            os.path.isdir(os.path.join(d, h)) for h in ("HSI", "hsi")
        ):
            return True
    return False


def search_roots() -> List[str]:
    """Base locations to search under, most specific first.

    DAETF_DATA_ROOTS (os.pathsep separated) always takes priority, so discovery
    can be overridden without editing any code.
    """
    roots: List[str] = []
    env = os.environ.get("DAETF_DATA_ROOTS", "")
    roots += [p for p in env.split(os.pathsep) if p]
    roots += ["/kaggle/input"]
    roots += [os.path.join(os.getcwd(), "data"), os.getcwd()]
    return [r for r in roots if os.path.isdir(r)]


def find_dataset_roots(base: str, max_depth: int = 5) -> List[str]:
    """Breadth-first search under `base` for directories exposing <split>/HSI.

    Kaggle does not mount datasets at a predictable depth: attaching
    `owner/cave-dataset-2` can appear as /kaggle/input/cave-dataset-2/Data or
    as /kaggle/input/datasets/owner/cave-dataset-2/Data depending on how the
    kernel was configured. Searching a fixed depth silently fails on the
    second layout, so walk until a dataset is found.

    Only directories are visited, HSI/RGB leaves are never descended into, and
    the search stops descending as soon as a root matches - so this stays cheap
    even when the tree holds thousands of .mat files.
    """
    found: List[str] = []
    queue: List[Tuple[str, int]] = [(base, 0)]
    seen = set()
    while queue:
        path, depth = queue.pop(0)
        real = os.path.realpath(path)
        if real in seen:
            continue
        seen.add(real)
        if _looks_like_dataset(path):
            found.append(path)
            continue                      # do not descend into a match
        if depth >= max_depth:
            continue
        try:
            for entry in sorted(os.scandir(path), key=lambda e: e.name):
                if entry.is_dir(follow_symlinks=False) and \
                        entry.name not in ("HSI", "hsi", "RGB", "rgb"):
                    queue.append((entry.path, depth + 1))
        except OSError:
            continue
    return found


def discover_dataset(hints: Sequence[str] = (), required: bool = True,
                     verbose: bool = True) -> Optional[str]:
    """Locate a dataset root whose path matches one of `hints`.

    Handles `<root>/Data/Train/HSI`, `<root>/Train/HSI` and arbitrarily nested
    Kaggle mount points.
    """
    found: List[str] = []
    for root in search_roots():
        for cand in find_dataset_roots(root):
            if cand not in found:
                found.append(cand)
    if hints:
        lowered = [h.lower() for h in hints]
        ranked = [f for f in found if any(h in f.lower() for h in lowered)]
        found = ranked or found
    if not found:
        if required:
            listing = []
            for r in search_roots():
                try:
                    listing.append(f"{r} -> {sorted(os.listdir(r))[:8]}")
                except OSError:
                    pass
            raise FileNotFoundError(
                f"no dataset matching {list(hints)} found.\n"
                f"Searched (depth 5) under:\n  " + "\n  ".join(listing) +
                "\nA dataset root must contain <split>/HSI, e.g. Data/Train/HSI.\n"
                "Set DAETF_DATA_ROOTS or pass Config(source_root=...) explicitly."
            )
        if verbose:
            print(f"[config] optional dataset {list(hints)} not found - skipping")
        return None
    if verbose:
        print(f"[config] using dataset root: {found[0]}")
    return found[0]


def available_splits(root: str) -> Dict[str, str]:
    """Map canonical split name -> the directory name actually present."""
    out = {}
    for canonical, names in (("Train", SPLIT_NAMES), ("Test", TEST_NAMES)):
        for n in names:
            if os.path.isdir(os.path.join(root, n)):
                out[canonical] = n
                break
    return out


def infer_channels(root: str) -> Tuple[int, int]:
    """Read one HSI/RGB pair and report their channel counts."""
    splits = available_splits(root)
    split = splits.get("Train") or splits.get("Test")
    if split is None:
        raise FileNotFoundError(f"no usable split under {root}")
    base = os.path.join(root, split)
    hsi_dir = next(os.path.join(base, d) for d in ("HSI", "hsi")
                   if os.path.isdir(os.path.join(base, d)))
    rgb_dir = next((os.path.join(base, d) for d in ("RGB", "rgb")
                    if os.path.isdir(os.path.join(base, d))), None)
    hsi = np.squeeze(load_mat(sorted(glob.glob(os.path.join(hsi_dir, "*.mat")))[0]))
    bands = int(min(hsi.shape))
    msi_bands = 3
    if rgb_dir:
        rgb = np.squeeze(load_mat(sorted(glob.glob(os.path.join(rgb_dir, "*.mat")))[0]))
        msi_bands = int(min(rgb.shape))
    return bands, msi_bands


def find_pairs(root: str, split: str) -> List[Tuple[str, str, str]]:
    """Matched (stem, hsi_path, rgb_path) triples for a canonical split name."""
    actual = available_splits(root).get(split, split)
    base = os.path.join(root, actual)
    hsi_dir = next((os.path.join(base, d) for d in ("HSI", "hsi")
                    if os.path.isdir(os.path.join(base, d))), None)
    rgb_dir = next((os.path.join(base, d) for d in ("RGB", "rgb")
                    if os.path.isdir(os.path.join(base, d))), None)
    if not hsi_dir or not rgb_dir:
        raise FileNotFoundError(f"no HSI/RGB folders under {base}")
    rgb = {os.path.splitext(os.path.basename(p))[0]: p
           for p in glob.glob(os.path.join(rgb_dir, "*.mat"))}
    out = []
    for h in sorted(glob.glob(os.path.join(hsi_dir, "*.mat"))):
        stem = os.path.splitext(os.path.basename(h))[0]
        if stem in rgb:
            out.append((stem, h, rgb[stem]))
    if not out:
        raise RuntimeError(f"no matched pairs under {base}")
    return out

In [ ]:
%%writefile hsifusion/config.py
"""Shared experiment configuration.

`BaseConfig` holds everything the evaluation protocol depends on - the scale
factor, the degradation, the metric settings, the optimiser schedule. Every
proposal subclasses it and adds only its own architectural fields.

That split is the point: because all four proposals inherit the same protocol
fields and the same defaults, a difference in their results is attributable to
the architecture rather than to one of them quietly evaluating at a different
scale factor or normalisation - which is exactly how the ten published
baselines in `existing/` became incomparable.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass
from typing import Optional, Sequence, Tuple

from .io_utils import discover_dataset, infer_channels


@dataclass
class BaseConfig:
    # --- data (None => auto-discover) --------------------------------------
    source_root: Optional[str] = None
    target_root: Optional[str] = None
    bands: Optional[int] = None
    msi_bands: Optional[int] = None

    # --- protocol: identical for every proposal ----------------------------
    scale: int = 4
    patch: int = 64
    blur_ksize: int = 9
    eval_sigma: float = 1.2
    sigma_range: Tuple[float, float] = (0.6, 2.4)
    aniso: float = 0.5
    noise_range: Tuple[float, float] = (0.0, 0.03)
    srf_jitter: float = 0.35

    # --- optimisation --------------------------------------------------------
    iters: int = 20000
    batch: int = 16
    lr: float = 2e-4
    min_lr: float = 1e-6
    warmup: int = 500
    grad_clip: float = 1.0
    amp: bool = True
    workers: int = 2
    seed: int = 42
    cache_limit: int = 12

    # --- shared loss weights -------------------------------------------------
    w_char: float = 1.0
    w_sam: float = 0.30
    w_grad: float = 0.20
    w_ssim: float = 0.15
    w_spat: float = 0.50
    w_spec: float = 0.50
    use_physics: bool = True

    # --- bookkeeping ---------------------------------------------------------
    out_dir: str = "./out"
    val_every: int = 1000
    log_every: int = 100
    val_scenes: int = 4
    name: str = "model"

    def __post_init__(self) -> None:
        assert self.patch % self.scale == 0, "patch must be divisible by scale"

    def resolve(self, source_hints: Sequence[str] = ("cave",),
                target_hints: Sequence[str] = ("harvard",),
                verbose: bool = True) -> "BaseConfig":
        """Fill in any field still None by inspecting the filesystem."""
        if self.source_root is None:
            self.source_root = discover_dataset(source_hints, verbose=verbose)
        if self.target_root is None:
            self.target_root = discover_dataset(target_hints, required=False,
                                                verbose=verbose)
        if self.bands is None or self.msi_bands is None:
            b, m = infer_channels(self.source_root)
            self.bands = self.bands or b
            self.msi_bands = self.msi_bands or m
            if verbose:
                print(f"[config] inferred bands={self.bands} msi_bands={self.msi_bands}")
        return self

    def to_dict(self) -> dict:
        return asdict(self)

In [ ]:
%%writefile hsifusion/degrade.py
"""Forward observation model: blur, decimation and the fixed evaluation operator.

Keeping the degradation differentiable is what allows the spatial-consistency
term of the loss to be back-propagated through, and therefore what allows
self-supervised adaptation on a domain with no ground truth.
"""

from __future__ import annotations

import math
from typing import TYPE_CHECKING, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

if TYPE_CHECKING:  # pragma: no cover
    from .config import Config


def gaussian_kernel2d(ksize: int, sx: float, sy: float, theta: float) -> torch.Tensor:
    """Rotated anisotropic Gaussian blur kernel, normalised to sum 1."""
    ax = torch.arange(ksize, dtype=torch.float32) - (ksize - 1) / 2.0
    yy, xx = torch.meshgrid(ax, ax, indexing="ij")
    cos_t, sin_t = math.cos(theta), math.sin(theta)
    xr = xx * cos_t + yy * sin_t
    yr = -xx * sin_t + yy * cos_t
    k = torch.exp(-0.5 * ((xr / sx) ** 2 + (yr / sy) ** 2))
    return k / k.sum().clamp_min(1e-12)


def blur_downsample(x: torch.Tensor, kernel: torch.Tensor, scale: int) -> torch.Tensor:
    """Apply a per-sample blur kernel then decimate. Differentiable.

    x      : [B, C, H, W]
    kernel : [k, k] (shared) or [B, k, k] (one kernel per sample)
    """
    b, c, _, _ = x.shape
    if kernel.dim() == 2:
        kernel = kernel.unsqueeze(0).expand(b, -1, -1)
    k = kernel.shape[-1]
    pad = k // 2
    # fold the batch into the channel axis so each sample keeps its own kernel
    w = kernel.to(x.dtype).reshape(b, 1, 1, k, k).expand(b, c, 1, k, k).reshape(b * c, 1, k, k)
    xr = x.reshape(1, b * c, *x.shape[-2:])
    xr = F.pad(xr, (pad, pad, pad, pad), mode="reflect")
    out = F.conv2d(xr, w, groups=b * c)
    out = out.reshape(b, c, *out.shape[-2:])
    return out[..., ::scale, ::scale].contiguous()


class FixedDegradation(nn.Module):
    """Non-learnable blur+decimate: builds the evaluation LR input and backs the
    spatial-consistency loss."""

    def __init__(self, scale: int, ksize: int = 9, sigma: float = 1.2):
        super().__init__()
        self.scale = scale
        self.register_buffer("kernel", gaussian_kernel2d(ksize, sigma, sigma, 0.0))

    @classmethod
    def from_config(cls, cfg: "Config") -> "FixedDegradation":
        return cls(cfg.scale, ksize=cfg.blur_ksize, sigma=cfg.eval_sigma)

    def forward(self, x: torch.Tensor, kernel: Optional[torch.Tensor] = None
                ) -> torch.Tensor:
        k = self.kernel if kernel is None else kernel
        return blur_downsample(x, k, self.scale)

In [ ]:
%%writefile hsifusion/metrics.py
"""One metric implementation, shared by every method and both datasets.

The v1 benchmark computed PSNR/SSIM/SAM/ERGAS separately inside each of the 20
notebooks, with different data ranges, different normalisations and ERGAS scale
factors that did not always match the actual downsampling factor. Those numbers
were therefore not comparable across methods. Everything here is fixed:

  * PSNR uses a constant data_range (default 1.0), never the per-image maximum,
    which otherwise inflates scores on dark scenes.
  * SSIM is Gaussian-windowed (11x11, sigma 1.5) and averaged over bands.
  * SAM is reported in degrees, ignoring degenerate zero-spectra pixels.
  * ERGAS receives the true scale factor of the experiment.
"""

from __future__ import annotations

from typing import Dict

import numpy as np
import torch
import torch.nn.functional as F


def _gauss_window(size: int, sigma: float, device, dtype) -> torch.Tensor:
    coords = torch.arange(size, device=device, dtype=dtype) - size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    return g[:, None] @ g[None, :]


def ssim_torch(pred: torch.Tensor, target: torch.Tensor, data_range: float = 1.0,
               size: int = 11, sigma: float = 1.5) -> torch.Tensor:
    """Gaussian-windowed SSIM, averaged over channels. Differentiable."""
    c = pred.shape[1]
    win = _gauss_window(size, sigma, pred.device, pred.dtype).expand(c, 1, size, size)
    mu1 = F.conv2d(pred, win, padding=size // 2, groups=c)
    mu2 = F.conv2d(target, win, padding=size // 2, groups=c)
    mu1s, mu2s, mu12 = mu1 ** 2, mu2 ** 2, mu1 * mu2
    s1 = F.conv2d(pred * pred, win, padding=size // 2, groups=c) - mu1s
    s2 = F.conv2d(target * target, win, padding=size // 2, groups=c) - mu2s
    s12 = F.conv2d(pred * target, win, padding=size // 2, groups=c) - mu12
    c1, c2 = (0.01 * data_range) ** 2, (0.03 * data_range) ** 2
    m = ((2 * mu12 + c1) * (2 * s12 + c2)) / ((mu1s + mu2s + c1) * (s1 + s2 + c2))
    return m.mean()


def _hwc(x: np.ndarray) -> np.ndarray:
    return x if x.shape[-1] <= 64 else np.transpose(x, (1, 2, 0))


def metric_psnr(pred: np.ndarray, ref: np.ndarray, data_range: float = 1.0) -> float:
    mse = float(np.mean((pred - ref) ** 2))
    return 99.0 if mse <= 1e-12 else float(10 * np.log10(data_range ** 2 / mse))


def metric_sam(pred: np.ndarray, ref: np.ndarray, eps: float = 1e-8) -> float:
    p, r = _hwc(pred).reshape(-1, pred.shape[-1]), _hwc(ref).reshape(-1, ref.shape[-1])
    cos = (p * r).sum(1) / np.maximum(np.linalg.norm(p, axis=1) * np.linalg.norm(r, axis=1), eps)
    ang = np.degrees(np.arccos(np.clip(cos, -1, 1)))
    return float(np.mean(ang[np.isfinite(ang)]))


def metric_ergas(pred: np.ndarray, ref: np.ndarray, scale: int, eps: float = 1e-8) -> float:
    p, r = _hwc(pred), _hwc(ref)
    rmse = np.sqrt(np.mean((p - r) ** 2, axis=(0, 1)))
    mu = np.maximum(np.mean(r, axis=(0, 1)), eps)
    return float(100.0 / scale * np.sqrt(np.mean((rmse / mu) ** 2)))


def metric_ssim(pred: np.ndarray, ref: np.ndarray, data_range: float = 1.0) -> float:
    p = torch.from_numpy(np.ascontiguousarray(_hwc(pred).transpose(2, 0, 1)))[None].float()
    r = torch.from_numpy(np.ascontiguousarray(_hwc(ref).transpose(2, 0, 1)))[None].float()
    return float(ssim_torch(p, r, data_range=data_range))


def evaluate_arrays(pred: np.ndarray, ref: np.ndarray, scale: int) -> Dict[str, float]:
    """The four reported metrics for one scene."""
    return {
        "psnr": metric_psnr(pred, ref),
        "ssim": metric_ssim(pred, ref),
        "sam": metric_sam(pred, ref),
        "ergas": metric_ergas(pred, ref, scale),
    }

In [ ]:
%%writefile hsifusion/losses.py
"""Shared loss terms and the common fidelity+physics objective.

Every proposal optimises the same base objective so that architectural
differences are not confounded with differences in what was optimised. A
proposal that needs extra terms (a rank penalty, an expert-balance penalty,
a per-stage supervision) adds them through `extra_terms`, which keeps the
shared part identical and the additions explicit.
"""

from __future__ import annotations

from typing import Callable, Dict, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

from .degrade import FixedDegradation
from .metrics import ssim_torch


def charbonnier(x: torch.Tensor, y: torch.Tensor, eps: float = 1e-3) -> torch.Tensor:
    """Robust L1: differentiable at zero, less outlier-sensitive than L2."""
    return torch.sqrt((x - y) ** 2 + eps ** 2).mean()


def sam_loss(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """Mean spectral angle in radians - the metric every baseline loses under
    domain shift, optimised directly."""
    p, t = pred.flatten(2), target.flatten(2)
    num = (p * t).sum(dim=1)
    den = p.norm(dim=1) * t.norm(dim=1)
    cos = (num / den.clamp_min(eps)).clamp(-1 + 1e-6, 1 - 1e-6)
    return torch.acos(cos).mean()


def gradient_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    dx_p = pred[..., :, 1:] - pred[..., :, :-1]
    dx_t = target[..., :, 1:] - target[..., :, :-1]
    dy_p = pred[..., 1:, :] - pred[..., :-1, :]
    dy_t = target[..., 1:, :] - target[..., :-1, :]
    return F.l1_loss(dx_p, dx_t) + F.l1_loss(dy_p, dy_t)


def mmd_rbf(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """Multi-bandwidth RBF MMD, bandwidth set from the median pairwise distance."""
    z = torch.cat([x, y], dim=0)
    d = torch.cdist(z, z) ** 2
    n = x.shape[0]
    med = d.detach().flatten().median().clamp_min(1e-6)
    k = sum(torch.exp(-d / (med * s)) for s in (0.25, 0.5, 1.0, 2.0, 4.0))
    return k[:n, :n].mean() + k[n:, n:].mean() - 2 * k[:n, n:].mean()


class FusionLoss(nn.Module):
    """Fidelity + physics, shared by every proposal.

        L = w_char C + w_sam SAM + w_grad G + w_ssim (1-SSIM)     [fidelity]
          + w_spat ||Down(Y) - LR|| + w_spec ||SRF(Y) - MSI||     [physics]
          + extra_terms(...)                                      [per-proposal]

    The physics terms need no ground truth, so `supervised=False` yields an
    objective that is computable on an unlabelled scene from an unseen sensor.
    That is what makes test-time adaptation and the fully self-supervised
    proposal possible with the same code.
    """

    def __init__(self, cfg, srf: torch.Tensor,
                 extra_terms: Optional[Callable] = None):
        super().__init__()
        self.cfg = cfg
        self.degrade = FixedDegradation(cfg.scale, ksize=cfg.blur_ksize,
                                        sigma=cfg.eval_sigma)
        self.register_buffer("srf", srf)          # [bands, msi_bands]
        self.extra_terms = extra_terms

    def apply_srf(self, x: torch.Tensor) -> torch.Tensor:
        w = self.srf.to(x.dtype).t().reshape(self.srf.shape[1], self.srf.shape[0], 1, 1)
        return F.conv2d(x, w)

    def forward(self, out: Dict[str, torch.Tensor], target: Optional[torch.Tensor],
                lr_hsi: torch.Tensor, msi: torch.Tensor, model: nn.Module,
                kernel: Optional[torch.Tensor] = None,
                supervised: bool = True,
                **kw) -> Tuple[torch.Tensor, Dict[str, float]]:
        cfg = self.cfg
        pred = out["out"]
        logs: Dict[str, float] = {}
        total = pred.new_zeros(())

        if supervised and target is not None:
            l_char = charbonnier(pred, target)
            l_sam = sam_loss(pred, target)
            l_grad = gradient_loss(pred, target)
            l_ssim = 1.0 - ssim_torch(pred.clamp(0, 1).float(), target.float())
            total = (total + cfg.w_char * l_char + cfg.w_sam * l_sam
                     + cfg.w_grad * l_grad + cfg.w_ssim * l_ssim)
            logs.update(char=l_char.item(), sam=l_sam.item(),
                        grad=l_grad.item(), ssim=l_ssim.item())

        if cfg.use_physics or not supervised:
            l_spat = charbonnier(self.degrade(pred, kernel), lr_hsi)
            l_spec = charbonnier(self.apply_srf(pred), msi)
            total = total + cfg.w_spat * l_spat + cfg.w_spec * l_spec
            logs.update(spat=l_spat.item(), spec=l_spec.item())

        if self.extra_terms is not None:
            extra, extra_logs = self.extra_terms(
                out=out, target=target, lr_hsi=lr_hsi, msi=msi, model=model,
                cfg=cfg, supervised=supervised, **kw)
            total = total + extra
            logs.update(extra_logs)

        logs["total"] = total.item()
        return total, logs

In [ ]:
%%writefile hsifusion/data.py
"""Datasets, scene caching and spectral-response estimation.

The v1 dataset returned `torch.randn(...)` with a hardcoded length of 100, so
nothing was ever trained on real data. This module loads the actual .mat scenes
and synthesises the observation pair through the physical forward model.
"""

from __future__ import annotations

import math
import random
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset

from .config import BaseConfig as Config
from .degrade import blur_downsample, gaussian_kernel2d
from .io_utils import find_pairs, load_mat, to_chw01


class SceneCache:
    """Bounded LRU cache of decoded scenes, held in float16.

    Harvard scenes are 1040x1392x31; caching them all as float32 would need
    ~5.4 GB, so scenes are stored halved and evicted least-recently-used.
    """

    def __init__(self, bands: int, msi_bands: int, limit: int = 12):
        self.bands, self.msi_bands, self.limit = bands, msi_bands, limit
        self.store: Dict[str, Tuple[np.ndarray, np.ndarray]] = {}
        self.order: List[str] = []

    def get(self, stem: str, hsi_path: str, rgb_path: str
            ) -> Tuple[np.ndarray, np.ndarray]:
        if stem in self.store:
            self.order.remove(stem)
            self.order.append(stem)
            return self.store[stem]
        hsi = to_chw01(load_mat(hsi_path), self.bands)
        rgb = to_chw01(load_mat(rgb_path), self.msi_bands)
        if rgb.shape[-2:] != hsi.shape[-2:]:
            t = torch.from_numpy(rgb)[None]
            rgb = F.interpolate(t, size=hsi.shape[-2:], mode="bicubic",
                                align_corners=False).clamp(0, 1)[0].numpy()
        item = (hsi.astype(np.float16), rgb.astype(np.float16))
        self.store[stem] = item
        self.order.append(stem)
        while len(self.order) > self.limit:
            self.store.pop(self.order.pop(0), None)
        return item


class FusionPatchDataset(Dataset):
    """Samples HR patches and synthesises (LR-HSI, MSI) through the forward
    model, randomising blur, noise and spectral response.

    The randomisation is the domain-shift defence: a model that has only ever
    seen one fixed bicubic degradation has no reason to work on a real sensor.
    """

    def __init__(self, root: str, split: str, cfg: Config, train: bool = True,
                 srf: Optional[np.ndarray] = None, length: int = 8000):
        self.cfg, self.train, self.length = cfg, train, length
        self.pairs = find_pairs(root, split)
        self.cache = SceneCache(
            cfg.bands, cfg.msi_bands,
            limit=min(len(self.pairs), cfg.cache_limit) if train else 4)
        self.srf = srf

    def __len__(self) -> int:
        return self.length if self.train else len(self.pairs)

    def _sample_kernel(self) -> Tuple[torch.Tensor, List[float]]:
        cfg = self.cfg
        sx = random.uniform(*cfg.sigma_range)
        sy = sx if random.random() > cfg.aniso else random.uniform(*cfg.sigma_range)
        th = random.uniform(0, math.pi)
        return (gaussian_kernel2d(cfg.blur_ksize, sx, sy, th),
                [sx, sy, math.sin(2 * th), math.cos(2 * th)])

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        cfg = self.cfg
        stem, hp, rp = (self.pairs[random.randrange(len(self.pairs))] if self.train
                        else self.pairs[idx % len(self.pairs)])
        hsi, rgb = self.cache.get(stem, hp, rp)

        if self.train:
            p = cfg.patch
            _, h, w = hsi.shape
            top, left = random.randrange(0, h - p + 1), random.randrange(0, w - p + 1)
            gt = torch.from_numpy(hsi[:, top:top + p, left:left + p].astype(np.float32))
            msi = torch.from_numpy(rgb[:, top:top + p, left:left + p].astype(np.float32))
            if random.random() < 0.5:
                gt, msi = torch.flip(gt, [-1]), torch.flip(msi, [-1])
            k = random.randrange(4)          # the p4 stem handles these natively
            if k:
                gt, msi = torch.rot90(gt, k, (-2, -1)), torch.rot90(msi, k, (-2, -1))
        else:
            p = (min(hsi.shape[1], hsi.shape[2]) // cfg.scale) * cfg.scale
            gt = torch.from_numpy(hsi[:, :p, :p].astype(np.float32))
            msi = torch.from_numpy(rgb[:, :p, :p].astype(np.float32))

        es = cfg.eval_sigma
        kernel, deg = (self._sample_kernel() if self.train else
                       (gaussian_kernel2d(cfg.blur_ksize, es, es, 0.0), [es, es, 0.0, 1.0]))

        lr = blur_downsample(gt[None], kernel, cfg.scale)[0]
        noise = random.uniform(*cfg.noise_range) if self.train else 0.0
        if noise > 0:
            lr = (lr + torch.randn_like(lr) * noise).clamp(0, 1)

        # sometimes replace the real RGB with a jittered synthetic MSI, so the
        # model never assumes one fixed spectral response function
        if self.train and self.srf is not None and random.random() < cfg.srf_jitter:
            s = torch.from_numpy(self.srf).float()
            s = (s * (1 + 0.15 * torch.randn_like(s))).clamp_min(0)
            s = s / s.sum(0, keepdim=True).clamp_min(1e-6) * float(self.srf.sum(0).mean())
            msi = torch.einsum("chw,cm->mhw", gt, s).clamp(0, 1)

        return {"lr": lr, "msi": msi, "gt": gt,
                "deg": torch.tensor(deg + [noise], dtype=torch.float32),
                "kernel": kernel, "name": stem}


def estimate_srf(root: str, split: str, cfg: Config, max_scenes: int = 8,
                 samples_per_scene: int = 20000) -> np.ndarray:
    """Least-squares spectral response function: min_S || HSI @ S - RGB ||^2.

    Recovering the SRF from the data makes the spectral-consistency loss a real
    physical constraint instead of a hand-picked approximation, and it adapts
    automatically to a dataset whose RGB was rendered with a different response.
    """
    pairs = find_pairs(root, split)[:max_scenes]
    xs, ys = [], []
    rng = np.random.default_rng(0)
    for stem, hp, rp in pairs:
        hsi = to_chw01(load_mat(hp), cfg.bands)
        rgb = to_chw01(load_mat(rp), cfg.msi_bands)
        if rgb.shape[-2:] != hsi.shape[-2:]:
            rgb = F.interpolate(torch.from_numpy(rgb)[None], size=hsi.shape[-2:],
                                mode="bicubic", align_corners=False)[0].numpy()
        h = hsi.reshape(cfg.bands, -1).T
        r = rgb.reshape(cfg.msi_bands, -1).T
        idx = rng.choice(h.shape[0], size=min(samples_per_scene, h.shape[0]),
                         replace=False)
        xs.append(h[idx])
        ys.append(r[idx])
    x = np.concatenate(xs).astype(np.float64)
    y = np.concatenate(ys).astype(np.float64)
    s, *_ = np.linalg.lstsq(x, y, rcond=None)
    return np.clip(s, 0.0, None).astype(np.float32)

In [ ]:
%%writefile hsifusion/engine.py
"""Training, tiled inference, evaluation and test-time adaptation.

Model-agnostic: anything with `forward(lr_hsi, msi) -> {"out": tensor}` and a
`.cfg` attribute can be trained and scored here. That contract is what lets all
four proposals share one protocol, one metric implementation and one set of
result tables.
"""

from __future__ import annotations

import json
import math
import os
import random
import time
from typing import Callable, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from .data import FusionPatchDataset, SceneCache, estimate_srf
from .degrade import FixedDegradation
from .io_utils import find_pairs
from .metrics import evaluate_arrays


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def cosine_lr(step: int, cfg) -> float:
    if step < cfg.warmup:
        return cfg.lr * step / max(cfg.warmup, 1)
    t = (step - cfg.warmup) / max(cfg.iters - cfg.warmup, 1)
    return cfg.min_lr + 0.5 * (cfg.lr - cfg.min_lr) * (1 + math.cos(math.pi * t))


def n_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


# ------------------------------------------------------------------- inference
@torch.no_grad()
def tiled_inference(model: nn.Module, lr: torch.Tensor, msi: torch.Tensor,
                    scale: int, bands: Optional[int] = None,
                    tile_hr: int = 256, overlap: int = 32) -> torch.Tensor:
    """Hann-weighted overlapping tiles, so full scenes fit in 16 GB and tile
    seams do not appear in the output."""
    model.eval()
    bands = bands or getattr(model, "cfg", None).bands
    bsz, _, h_hr, w_hr = msi.shape
    tile_lr, ov_lr = tile_hr // scale, overlap // scale
    tile_lr = min(tile_lr, lr.shape[2], lr.shape[3])
    tile_hr = tile_lr * scale
    step_lr = max(tile_lr - ov_lr, 1)
    out = torch.zeros(bsz, bands, h_hr, w_hr, device=lr.device, dtype=torch.float32)
    wsum = torch.zeros(bsz, 1, h_hr, w_hr, device=lr.device, dtype=torch.float32)

    win1d = torch.hann_window(tile_hr, periodic=False, device=lr.device).clamp_min(1e-3)
    win = (win1d[:, None] * win1d[None, :])[None, None]

    ys = list(range(0, max(lr.shape[2] - tile_lr, 0) + 1, step_lr))
    xs = list(range(0, max(lr.shape[3] - tile_lr, 0) + 1, step_lr))
    if ys[-1] + tile_lr < lr.shape[2]:
        ys.append(lr.shape[2] - tile_lr)
    if xs[-1] + tile_lr < lr.shape[3]:
        xs.append(lr.shape[3] - tile_lr)

    for y0 in ys:
        for x0 in xs:
            y1, x1 = y0 + tile_lr, x0 + tile_lr
            hy0, hx0, hy1, hx1 = y0 * scale, x0 * scale, y1 * scale, x1 * scale
            pred = model(lr[:, :, y0:y1, x0:x1],
                         msi[:, :, hy0:hy1, hx0:hx1])["out"].float()
            w = win[..., :pred.shape[-2], :pred.shape[-1]]
            out[:, :, hy0:hy1, hx0:hx1] += pred * w
            wsum[:, :, hy0:hy1, hx0:hx1] += w
    return (out / wsum.clamp_min(1e-6)).clamp(0, 1)


@torch.no_grad()
def evaluate_dataset(model: nn.Module, root: str, cfg, split: str = "Test",
                     device: str = "cuda", limit: Optional[int] = None,
                     tile_hr: int = 256, verbose: bool = True,
                     return_rows: bool = False):
    """Full-scene evaluation through the unified metric module."""
    pairs = find_pairs(root, split)
    if limit:
        pairs = pairs[:limit]
    cache = SceneCache(cfg.bands, cfg.msi_bands, limit=2)
    degrade = FixedDegradation(cfg.scale, cfg.blur_ksize, cfg.eval_sigma).to(device)
    rows, agg = [], {"psnr": [], "ssim": [], "sam": [], "ergas": []}

    for stem, hp, rp in pairs:
        hsi, rgb = cache.get(stem, hp, rp)
        h = (hsi.shape[1] // cfg.scale) * cfg.scale
        w = (hsi.shape[2] // cfg.scale) * cfg.scale
        gt = torch.from_numpy(hsi[:, :h, :w].astype(np.float32))[None].to(device)
        msi = torch.from_numpy(rgb[:, :h, :w].astype(np.float32))[None].to(device)
        lr = degrade(gt)
        pred = tiled_inference(model, lr, msi, cfg.scale, cfg.bands, tile_hr=tile_hr)
        m = evaluate_arrays(pred[0].cpu().numpy().transpose(1, 2, 0),
                            gt[0].cpu().numpy().transpose(1, 2, 0), cfg.scale)
        rows.append({"scene": stem, **m})
        for k, v in m.items():
            agg[k].append(v)
        if verbose:
            print(f"  {stem:<24} PSNR={m['psnr']:7.3f}  SSIM={m['ssim']:.4f}  "
                  f"SAM={m['sam']:6.3f}  ERGAS={m['ergas']:8.3f}")
        del gt, msi, lr, pred
        if device == "cuda":
            torch.cuda.empty_cache()

    mean = {k: float(np.mean(v)) for k, v in agg.items()}
    if verbose:
        print(f"  {'MEAN':<24} PSNR={mean['psnr']:7.3f}  SSIM={mean['ssim']:.4f}  "
              f"SAM={mean['sam']:6.3f}  ERGAS={mean['ergas']:8.3f}")
    return (mean, rows) if return_rows else mean


# -------------------------------------------------------------------- training
def train(cfg, build_model: Callable[[object], nn.Module],
          build_loss: Callable[[object, np.ndarray], nn.Module],
          device: str = "cuda", align_target: bool = False,
          feature_fn: Optional[Callable] = None, log_fn=print
          ) -> Tuple[nn.Module, Dict]:
    """Train any model that follows the (lr_hsi, msi) -> {'out': ...} contract.

    `align_target` draws unlabelled target-domain patches alongside and expects
    `feature_fn(model, lr, msi)` to return pooled features for an MMD penalty.
    No target ground truth is ever read, so cross-domain evaluation stays honest.
    """
    cfg.resolve(verbose=False)
    set_seed(cfg.seed)
    os.makedirs(cfg.out_dir, exist_ok=True)

    log_fn("estimating SRF from the training pairs ...")
    srf = estimate_srf(cfg.source_root, "Train", cfg)
    log_fn(f"SRF shape {srf.shape}, column sums {srf.sum(0).round(3).tolist()}")

    train_set = FusionPatchDataset(cfg.source_root, "Train", cfg, train=True, srf=srf,
                                   length=cfg.iters * cfg.batch)
    loader = DataLoader(train_set, batch_size=cfg.batch, shuffle=False,
                        num_workers=cfg.workers, pin_memory=(device == "cuda"),
                        drop_last=True, persistent_workers=cfg.workers > 0)

    tgt_loader = None
    if align_target and cfg.target_root and feature_fn is not None:
        tgt_set = FusionPatchDataset(cfg.target_root, "Train", cfg, train=True, srf=srf,
                                     length=cfg.iters * cfg.batch)
        tgt_loader = iter(DataLoader(tgt_set, batch_size=cfg.batch, shuffle=False,
                                     num_workers=max(1, cfg.workers // 2), drop_last=True))
        log_fn(f"domain alignment enabled against {cfg.target_root} (unlabelled)")

    model = build_model(cfg).to(device)
    crit = build_loss(cfg, srf).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=1e-5,
                            betas=(0.9, 0.99))
    use_amp = cfg.amp and device == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    total_params = n_params(model)
    log_fn(f"{cfg.name}: {total_params / 1e6:.2f} M parameters")

    history: Dict[str, list] = {"iter": [], "loss": [], "val": [],
                                "cfg": cfg.to_dict(), "params": total_params}
    best, t0 = -1e9, time.time()

    start_step = 1
    ckpt_path = os.path.join(cfg.out_dir, f"{getattr(cfg, 'name', 'model')}_checkpoint.pth")
    if os.path.exists(ckpt_path):
        log_fn(f"Resuming from {ckpt_path}")
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt["model"])
        if "ema_model" in locals() and "ema_model" in ckpt:
            ema_model.load_state_dict(ckpt["ema_model"])
        opt.load_state_dict(ckpt["opt"])
        scaler.load_state_dict(ckpt["scaler"])
        start_step = ckpt["step"] + 1
        best = ckpt.get("best", -1e9)
        history = ckpt.get("history", history)

    model.train()
    for step, batch in enumerate(loader, start=start_step):
        if step > cfg.iters:
            break
        for g in opt.param_groups:
            g["lr"] = cosine_lr(step, cfg)

        lr_hsi = batch["lr"].to(device, non_blocking=True)
        msi = batch["msi"].to(device, non_blocking=True)
        gt = batch["gt"].to(device, non_blocking=True)
        kernel = batch["kernel"].to(device, non_blocking=True)
        extra = {"deg_gt": batch["deg"].to(device, non_blocking=True)}

        if tgt_loader is not None:
            tb = next(tgt_loader)
            with torch.amp.autocast("cuda", enabled=use_amp):
                extra["tgt_feat"] = feature_fn(model, tb["lr"].to(device),
                                               tb["msi"].to(device))

        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=use_amp):
            out = model(lr_hsi, msi)
            loss, logs = crit(out, gt, lr_hsi, msi, model, kernel=kernel, **extra)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(opt)
        scaler.update()

        if step % cfg.log_every == 0:
            rate = step / (time.time() - t0)
            eta = (cfg.iters - step) / max(rate, 1e-6) / 60
            log_fn(f"it {step:6d}/{cfg.iters}  loss {logs['total']:.4f}  "
                   f"char {logs.get('char', 0):.4f}  sam {logs.get('sam', 0):.4f}  "
                   f"spat {logs.get('spat', 0):.4f}  spec {logs.get('spec', 0):.4f}  "
                   f"lr {opt.param_groups[0]['lr']:.2e}  {rate:.2f} it/s  eta {eta:.0f}m")
            history["iter"].append(step)
            history["loss"].append(logs["total"])

            save_dict = {
                "model": model.state_dict(),
                "opt": opt.state_dict(),
                "scaler": scaler.state_dict(),
                "step": step,
                "best": best,
                "history": history,
            }
            if "ema_model" in locals():
                save_dict["ema_model"] = locals()["ema_model"].state_dict()
            torch.save(save_dict, ckpt_path)

        if step % cfg.val_every == 0 or step == cfg.iters:
            m = evaluate_dataset(model, cfg.source_root, cfg, "Test", device,
                                 limit=cfg.val_scenes, verbose=False)
            log_fn(f"  [val@{step}] PSNR {m['psnr']:.3f}  SAM {m['sam']:.3f}  "
                   f"ERGAS {m['ergas']:.3f}")
            history["val"].append({"iter": step, **m})
            if m["psnr"] > best:
                best = m["psnr"]
                torch.save({"model": model.state_dict(), "cfg": cfg.to_dict(),
                            "srf": srf, "val": m},
                           os.path.join(cfg.out_dir, f"{cfg.name}_best.pth"))
            model.train()

    torch.save({"model": model.state_dict(), "cfg": cfg.to_dict(), "srf": srf,
                "params": total_params},
               os.path.join(cfg.out_dir, f"{cfg.name}_final.pth"))
    with open(os.path.join(cfg.out_dir, "history.json"), "w") as f:
        json.dump(history, f, indent=1)
    return model, history


# ------------------------------------------------------- test-time adaptation
@torch.no_grad()
def clone_state(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {k: v.detach().clone() for k, v in model.state_dict().items()}


def test_time_adapt(model: nn.Module, lr: torch.Tensor, msi: torch.Tensor,
                    crit: nn.Module, steps: int = 30, lr_rate: float = 5e-5,
                    restore: bool = True,
                    param_filter: Tuple[str, ...] = ("deg", "film", "gate", "fdrm")
                    ) -> torch.Tensor:
    """Self-supervised adaptation on one unlabelled scene, using only the
    physics terms. Adapting a subset of parameters keeps it cheap and stable."""
    state = clone_state(model) if restore else None
    model.train()
    params = [p for n, p in model.named_parameters()
              if any(k in n for k in param_filter)]
    if not params:                       # model exposes no conditioning params
        params = list(model.parameters())
    opt = torch.optim.Adam(params, lr=lr_rate)
    for _ in range(steps):
        opt.zero_grad(set_to_none=True)
        out = model(lr, msi)
        loss, _ = crit(out, None, lr, msi, model, supervised=False)
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        pred = model(lr, msi)["out"].clamp(0, 1)
    if state is not None:
        model.load_state_dict(state)
    return pred


@torch.no_grad()
def evaluate_with_tta(model: nn.Module, root: str, cfg, crit: nn.Module,
                      split: str = "Test", device: str = "cuda",
                      steps: int = 30, limit: Optional[int] = None,
                      tile_hr: int = 256, verbose: bool = True):
    """Cross-domain evaluation where each scene is adapted before scoring.

    Every scene restarts from the same trained weights: adapting cumulatively
    would make the result depend on the order the scenes are listed in and let
    information leak between test scenes.
    """
    pairs = find_pairs(root, split)
    if limit:
        pairs = pairs[:limit]
    cache = SceneCache(cfg.bands, cfg.msi_bands, limit=2)
    degrade = FixedDegradation(cfg.scale, cfg.blur_ksize, cfg.eval_sigma).to(device)
    rows, agg = [], {"psnr": [], "ssim": [], "sam": [], "ergas": []}
    base_state = clone_state(model)

    for stem, hp, rp in pairs:
        model.load_state_dict(base_state)
        hsi, rgb = cache.get(stem, hp, rp)
        h = (hsi.shape[1] // cfg.scale) * cfg.scale
        w = (hsi.shape[2] // cfg.scale) * cfg.scale
        ch, cw = min(h, tile_hr * 2), min(w, tile_hr * 2)
        oy, ox = (h - ch) // 2, (w - cw) // 2
        gt = torch.from_numpy(hsi[:, :h, :w].astype(np.float32))[None].to(device)
        msi = torch.from_numpy(rgb[:, :h, :w].astype(np.float32))[None].to(device)
        lr = degrade(gt)
        crop_lr = lr[:, :, oy // cfg.scale:(oy + ch) // cfg.scale,
                     ox // cfg.scale:(ox + cw) // cfg.scale]
        crop_msi = msi[:, :, oy:oy + ch, ox:ox + cw]
        with torch.enable_grad():
            test_time_adapt(model, crop_lr, crop_msi, crit, steps=steps, restore=False)
        pred = tiled_inference(model, lr, msi, cfg.scale, cfg.bands, tile_hr=tile_hr)
        m = evaluate_arrays(pred[0].cpu().numpy().transpose(1, 2, 0),
                            gt[0].cpu().numpy().transpose(1, 2, 0), cfg.scale)
        rows.append({"scene": stem, **m})
        for k, v in m.items():
            agg[k].append(v)
        if verbose:
            print(f"  {stem:<24} PSNR={m['psnr']:7.3f}  SSIM={m['ssim']:.4f}  "
                  f"SAM={m['sam']:6.3f}  ERGAS={m['ergas']:8.3f}")
        del gt, msi, lr, pred
        if device == "cuda":
            torch.cuda.empty_cache()

    model.load_state_dict(base_state)
    mean = {k: float(np.mean(v)) for k, v in agg.items()}
    if verbose:
        print(f"  {'MEAN (TTA)':<24} PSNR={mean['psnr']:7.3f}  SSIM={mean['ssim']:.4f}  "
              f"SAM={mean['sam']:6.3f}  ERGAS={mean['ergas']:8.3f}")
    return mean, rows


In [ ]:
%%writefile hsifusion/baselines.py
"""Same-protocol reference methods.

The ten deep baselines in `existing/` were each run under their own protocol -
different scale factors, normalisations and metric implementations - so none of
their numbers can be compared directly against ours. Re-running all ten under
one protocol needs their checkpoints and three incompatible frameworks.

These classical methods need no checkpoints and no training, so they can be run
through the *identical* pipeline: same degradation, same scale factor, same
metric module, same scenes. That gives the paper a set of rows that are
genuinely comparable today, and a floor that any learned method must clear.

  bicubic       interpolation only, ignores the MSI entirely - the lower bound
                that reveals how much of a score comes from the HSI alone
  gsa           Gram-Schmidt Adaptive component substitution, the classical
                pansharpening approach with per-band regression gains
  subspace_ls   coupled subspace estimator: a spectral basis from the LR-HSI,
                abundances solved in closed form against the MSI with Tikhonov
                regularisation toward the upsampled HSI

`subspace_ls` is the strongest of the three and is the classical family that
model-based deep unfolding methods descend from, so it is the meaningful
non-learned comparison.
"""

from __future__ import annotations

from typing import Callable, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn.functional as F

from .config import BaseConfig as Config
from .data import SceneCache
from .degrade import FixedDegradation
from .io_utils import find_pairs
from .metrics import evaluate_arrays


def _upsample(lr: torch.Tensor, scale: int, mode: str = "bicubic") -> torch.Tensor:
    return F.interpolate(lr, scale_factor=scale, mode=mode,
                         align_corners=False).clamp(0, 1)


def bicubic(lr_hsi: torch.Tensor, msi: torch.Tensor, srf: torch.Tensor,
            scale: int) -> torch.Tensor:
    """Interpolation only. Deliberately ignores the MSI."""
    return _upsample(lr_hsi, scale)


def gsa(lr_hsi: torch.Tensor, msi: torch.Tensor, srf: torch.Tensor,
        scale: int) -> torch.Tensor:
    """Gram-Schmidt Adaptive component substitution.

    Builds a synthetic low-resolution intensity from the upsampled HSI, then
    injects the detail the MSI carries, band by band, with gains from a
    least-squares regression of each band on that intensity.
    """
    up = _upsample(lr_hsi, scale)                       # [B,C,H,W]
    pan = msi.mean(dim=1, keepdim=True)                 # [B,1,H,W]

    b, c, h, w = up.shape
    x = up.reshape(b, c, -1)
    p = pan.reshape(b, 1, -1)

    # synthetic intensity: least-squares combination of HSI bands matching pan
    xt = x.transpose(1, 2)                              # [B,N,C]
    gram = xt.transpose(1, 2) @ xt                      # [B,C,C]
    rhs = xt.transpose(1, 2) @ p.transpose(1, 2)        # [B,C,1]
    eye = torch.eye(c, device=x.device, dtype=x.dtype)[None] * 1e-6
    coef = torch.linalg.solve(gram + eye, rhs)          # [B,C,1]
    inten = (coef.transpose(1, 2) @ x)                  # [B,1,N]

    det = p - inten
    iv = inten - inten.mean(dim=2, keepdim=True)
    var = (iv * iv).mean(dim=2, keepdim=True).clamp_min(1e-8)
    xv = x - x.mean(dim=2, keepdim=True)
    gain = (xv * iv).mean(dim=2, keepdim=True) / var    # [B,C,1]

    out = (x + gain * det).reshape(b, c, h, w)
    return out.clamp(0, 1)


def subspace_ls(lr_hsi: torch.Tensor, msi: torch.Tensor, srf: torch.Tensor,
                scale: int, rank: int = 8, lam: float = 0.15) -> torch.Tensor:
    """Coupled subspace estimator (closed form).

    Hyperspectral cubes are close to low rank: a handful of spectral basis
    vectors explain almost all the variance. Take that basis E from the LR-HSI
    by SVD, write the high-resolution image as X = E A, and solve for the
    abundances A using the MSI, regularised toward the abundances implied by
    the upsampled HSI:

        min_A  ||(Sᵀ E) A − Y_msi||²  +  λ ||A − A₀||²

    which has the closed-form solution

        A = (MᵀM + λI)⁻¹ (Mᵀ Y_msi + λ A₀),    M = Sᵀ E

    The MSI supplies spatial detail, the LR-HSI supplies spectral truth, and λ
    sets the balance. Without the regulariser the system is underdetermined
    whenever the subspace rank exceeds the MSI band count.
    """
    b, c, _, _ = lr_hsi.shape
    up = _upsample(lr_hsi, scale)                       # [B,C,H,W]
    _, _, h, w = up.shape
    out = torch.empty_like(up)

    for i in range(b):
        y = lr_hsi[i].reshape(c, -1).double()           # [C, n]
        # spectral subspace from the low-resolution cube (uncentred: the mean
        # spectrum is signal here, not a nuisance offset)
        u, _, _ = torch.linalg.svd(y @ y.t(), full_matrices=False)
        e = u[:, :rank]                                 # [C, r]

        s = srf.to(y.dtype).to(y.device)                # [C, m]
        m = s.t() @ e                                   # [m, r]
        ym = msi[i].reshape(msi.shape[1], -1).double()  # [m, N]
        a0 = e.t() @ up[i].reshape(c, -1).double()      # [r, N]

        lhs = m.t() @ m + lam * torch.eye(rank, dtype=y.dtype, device=y.device)
        rhs = m.t() @ ym + lam * a0
        a = torch.linalg.solve(lhs, rhs)                # [r, N]
        out[i] = (e @ a).reshape(c, h, w).to(out.dtype)

    return out.clamp(0, 1)


BASELINES: Dict[str, Callable] = {
    "Bicubic": bicubic,
    "GSA": gsa,
    "Subspace-LS": subspace_ls,
}


@torch.no_grad()
def evaluate_baseline(name: str, root: str, cfg: Config, srf: np.ndarray,
                      split: str = "Test", device: str = "cuda",
                      limit: Optional[int] = None, verbose: bool = True
                      ) -> Tuple[Dict[str, float], List[Dict]]:
    """Run one classical baseline through the identical evaluation pipeline."""
    fn = BASELINES[name]
    pairs = find_pairs(root, split)
    if limit:
        pairs = pairs[:limit]
    cache = SceneCache(cfg.bands, cfg.msi_bands, limit=2)
    degrade = FixedDegradation.from_config(cfg).to(device)
    srf_t = torch.from_numpy(srf).to(device)
    rows, agg = [], {"psnr": [], "ssim": [], "sam": [], "ergas": []}

    for stem, hp, rp in pairs:
        hsi, rgb = cache.get(stem, hp, rp)
        h = (hsi.shape[1] // cfg.scale) * cfg.scale
        w = (hsi.shape[2] // cfg.scale) * cfg.scale
        gt = torch.from_numpy(hsi[:, :h, :w].astype(np.float32))[None].to(device)
        msi = torch.from_numpy(rgb[:, :h, :w].astype(np.float32))[None].to(device)
        lr = degrade(gt)
        pred = fn(lr, msi, srf_t, cfg.scale).float()
        m = evaluate_arrays(pred[0].cpu().numpy().transpose(1, 2, 0),
                            gt[0].cpu().numpy().transpose(1, 2, 0), cfg.scale)
        rows.append({"scene": stem, **m})
        for k, v in m.items():
            agg[k].append(v)
        if verbose:
            print(f"  {stem:<24} PSNR={m['psnr']:7.3f}  SSIM={m['ssim']:.4f}  "
                  f"SAM={m['sam']:6.3f}  ERGAS={m['ergas']:8.3f}")
        del gt, msi, lr, pred
        if device == "cuda":
            torch.cuda.empty_cache()

    mean = {k: float(np.mean(v)) for k, v in agg.items()}
    if verbose:
        print(f"  {name + ' MEAN':<24} PSNR={mean['psnr']:7.3f}  "
              f"SSIM={mean['ssim']:.4f}  SAM={mean['sam']:6.3f}  "
              f"ERGAS={mean['ergas']:8.3f}")
    return mean, rows


@torch.no_grad()
def evaluate_all_baselines(root: str, cfg: Config, srf: np.ndarray,
                           split: str = "Test", device: str = "cuda",
                           limit: Optional[int] = None, verbose: bool = True
                           ) -> Dict[str, Dict]:
    """Every classical baseline on one dataset, under the unified protocol."""
    out = {}
    for name in BASELINES:
        if verbose:
            print(f"\n--- {name} ---")
        mean, rows = evaluate_baseline(name, root, cfg, srf, split, device,
                                       limit=limit, verbose=verbose)
        out[name] = {"mean": mean, "rows": rows}
    return out

In [ ]:
%%writefile hsifusion/experiments.py
"""Experiment harness: ablations, efficiency profiling, statistics and tables.

What separates a Q1 submission from a demo is not the architecture, it is the
evidence around it. This module produces:

  * per-scene results, not just means, so comparisons can be paired
  * a paired Wilcoxon signed-rank test and Cohen's d against every baseline
  * bootstrap 95% confidence intervals on each mean
  * a component ablation with matched control arms
  * multi-seed repeats, reported as mean +/- std
  * cost accounting: parameters, GFLOPs, latency, peak GPU memory
  * cross-domain transfer with and without test-time adaptation
  * scale-factor generalisation
  * Markdown and LaTeX tables ready to paste into a manuscript

Dependencies are numpy/torch only; the statistics are implemented directly so
the module runs on a bare Kaggle image without scipy.stats.
"""

from __future__ import annotations

import copy
import json
import os
import time
from typing import Callable, Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn

from .engine import (evaluate_dataset, evaluate_with_tta, n_params, set_seed,
                     tiled_inference, train)

METRICS = ("psnr", "ssim", "sam", "ergas")
HIGHER_IS_BETTER = {"psnr": True, "ssim": True, "sam": False, "ergas": False}


# ---------------------------------------------------------------- statistics
def bootstrap_ci(values: Sequence[float], n_boot: int = 10000, alpha: float = 0.05,
                 seed: int = 0) -> Tuple[float, float]:
    """Percentile bootstrap confidence interval for the mean."""
    v = np.asarray(values, dtype=np.float64)
    if v.size < 2:
        return (float(v.mean()) if v.size else float("nan"),) * 2
    rng = np.random.default_rng(seed)
    means = rng.choice(v, size=(n_boot, v.size), replace=True).mean(axis=1)
    return float(np.percentile(means, 100 * alpha / 2)), \
        float(np.percentile(means, 100 * (1 - alpha / 2)))


def _normal_sf(z: float) -> float:
    """Upper-tail standard normal probability via the error function."""
    return 0.5 * math_erfc(z / (2 ** 0.5))


def math_erfc(x: float) -> float:
    import math
    return math.erfc(x)


def wilcoxon_signed_rank(a: Sequence[float], b: Sequence[float]) -> Dict[str, float]:
    """Paired Wilcoxon signed-rank test with a normal approximation.

    Paired over scenes: every method is scored on the same scenes, so pairing is
    the correct design and is far more sensitive than an unpaired test on the
    10-20 scenes these datasets provide.
    """
    a, b = np.asarray(a, dtype=np.float64), np.asarray(b, dtype=np.float64)
    d = a - b
    d = d[d != 0]
    n = d.size
    if n < 1:
        return {"n": 0, "W": float("nan"), "z": float("nan"), "p": float("nan")}
    order = np.argsort(np.abs(d))
    ranks = np.empty(n, dtype=np.float64)
    ranks[order] = np.arange(1, n + 1)
    # average ranks within ties of |d|
    absd = np.abs(d)[order]
    i = 0
    while i < n:
        j = i
        while j + 1 < n and absd[j + 1] == absd[i]:
            j += 1
        if j > i:
            ranks[order[i:j + 1]] = np.mean(np.arange(i + 1, j + 2))
        i = j + 1
    w_pos = ranks[d > 0].sum()
    w_neg = ranks[d < 0].sum()
    w = min(w_pos, w_neg)
    mu = n * (n + 1) / 4.0
    sigma = (n * (n + 1) * (2 * n + 1) / 24.0) ** 0.5
    z = (w - mu) / sigma if sigma > 0 else 0.0
    p = 2 * _normal_sf(abs(z))
    return {"n": float(n), "W": float(w), "z": float(z), "p": float(min(p, 1.0))}


def cohens_d(a: Sequence[float], b: Sequence[float]) -> float:
    """Paired Cohen's d (mean difference over the sd of the differences)."""
    d = np.asarray(a, dtype=np.float64) - np.asarray(b, dtype=np.float64)
    sd = d.std(ddof=1)
    return float(d.mean() / sd) if sd > 0 else float("inf") if d.mean() else 0.0


def compare_methods(ours: List[Dict], theirs: List[Dict],
                    name_a: str = "ours", name_b: str = "baseline") -> Dict:
    """Paired comparison over the scenes both methods were scored on."""
    by_b = {r["scene"]: r for r in theirs}
    shared = [r for r in ours if r["scene"] in by_b]
    out = {"name_a": name_a, "name_b": name_b, "n_scenes": len(shared)}
    for m in METRICS:
        a = [r[m] for r in shared]
        b = [by_b[r["scene"]][m] for r in shared]
        test = wilcoxon_signed_rank(a, b)
        delta = float(np.mean(a) - np.mean(b))
        improved = delta > 0 if HIGHER_IS_BETTER[m] else delta < 0
        out[m] = {"mean_a": float(np.mean(a)), "mean_b": float(np.mean(b)),
                  "delta": delta, "improved": bool(improved),
                  "p": test["p"], "z": test["z"], "d": cohens_d(a, b),
                  "ci_a": bootstrap_ci(a), "ci_b": bootstrap_ci(b)}
    return out


def summarise_rows(rows: List[Dict]) -> Dict[str, Dict[str, float]]:
    """Mean, std and bootstrap CI for each metric across scenes."""
    out = {}
    for m in METRICS:
        v = [r[m] for r in rows]
        lo, hi = bootstrap_ci(v)
        out[m] = {"mean": float(np.mean(v)), "std": float(np.std(v, ddof=1)) if len(v) > 1 else 0.0,
                  "ci_lo": lo, "ci_hi": hi, "n": len(v)}
    return out


# ------------------------------------------------------------------ efficiency
def count_flops(model: nn.Module, lr_shape: Tuple[int, ...],
                msi_shape: Tuple[int, ...], device: str = "cpu") -> float:
    """Multiply-accumulate count for conv and linear layers, in GFLOPs.

    Implemented with forward hooks rather than an external dependency, so the
    number can be reported from a bare Kaggle image. Counts 2 FLOPs per MAC.
    """
    total = [0]

    def conv_hook(m, inp, out):
        out_elems = out.numel()
        k = m.weight.shape[2] * m.weight.shape[3]
        total[0] += 2 * out_elems * (m.in_channels // m.groups) * k

    def deconv_hook(m, inp, out):
        k = m.weight.shape[2] * m.weight.shape[3]
        total[0] += 2 * inp[0].numel() * (m.out_channels // m.groups) * k

    def lin_hook(m, inp, out):
        total[0] += 2 * out.numel() * m.in_features

    handles = []
    for mod in model.modules():
        if isinstance(mod, nn.Conv2d):
            handles.append(mod.register_forward_hook(conv_hook))
        elif isinstance(mod, nn.ConvTranspose2d):
            handles.append(mod.register_forward_hook(deconv_hook))
        elif isinstance(mod, nn.Linear):
            handles.append(mod.register_forward_hook(lin_hook))

    model.eval()
    with torch.no_grad():
        model(torch.zeros(*lr_shape, device=device), torch.zeros(*msi_shape, device=device))
    for h in handles:
        h.remove()
    return total[0] / 1e9


@torch.no_grad()
def profile_model(model, cfg, device: str = "cuda",
                  hr: int = 512, warmup: int = 3, runs: int = 10) -> Dict[str, float]:
    """Parameters, GFLOPs, latency and peak memory for one full scene."""
    lr_shape = (1, cfg.bands, hr // cfg.scale, hr // cfg.scale)
    msi_shape = (1, cfg.msi_bands, hr, hr)
    gflops = count_flops(copy.deepcopy(model).to("cpu"), lr_shape, msi_shape, "cpu")

    model = model.to(device).eval()
    lr = torch.zeros(*lr_shape, device=device)
    msi = torch.zeros(*msi_shape, device=device)
    for _ in range(warmup):
        tiled_inference(model, lr, msi, cfg.scale)
    if device == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    for _ in range(runs):
        tiled_inference(model, lr, msi, cfg.scale)
    if device == "cuda":
        torch.cuda.synchronize()
    dt = (time.time() - t0) / runs
    peak = torch.cuda.max_memory_allocated() / 2 ** 20 if device == "cuda" else float("nan")
    return {"params_M": n_params(model) / 1e6, "gflops": gflops,
            "latency_s": dt, "peak_mem_MB": peak, "hr": hr}


# ----------------------------------------------------------------------- tables
def markdown_table(headers: Sequence[str], rows: Sequence[Sequence]) -> str:
    head = "| " + " | ".join(str(h) for h in headers) + " |"
    sep = "|" + "|".join("---" for _ in headers) + "|"
    body = "\n".join("| " + " | ".join(str(c) for c in r) + " |" for r in rows)
    return "\n".join([head, sep, body])


def latex_table(headers: Sequence[str], rows: Sequence[Sequence],
                caption: str = "", label: str = "") -> str:
    cols = "l" + "c" * (len(headers) - 1)
    lines = [r"\begin{table}[t]", r"\centering",
             rf"\caption{{{caption}}}", rf"\label{{{label}}}",
             rf"\begin{{tabular}}{{{cols}}}", r"\toprule",
             " & ".join(str(h) for h in headers) + r" \\", r"\midrule"]
    lines += [" & ".join(str(c) for c in r) + r" \\" for r in rows]
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    return "\n".join(lines)


def comparison_table(entries: Dict[str, Dict[str, float]], fmt: str = "markdown",
                     caption: str = "", label: str = "") -> str:
    """entries: {method name -> {psnr, ssim, sam, ergas}} rendered with the
    best value in each column marked."""
    headers = ["Method", "PSNR (dB) up", "SSIM up", "SAM (deg) down", "ERGAS down"]
    best = {m: (max if HIGHER_IS_BETTER[m] else min)(
        e[m] for e in entries.values() if m in e) for m in METRICS}
    rows = []
    for name, e in entries.items():
        cells = [name]
        for m, prec in zip(METRICS, (3, 4, 3, 3)):
            val = e.get(m)
            if val is None:
                cells.append("-")
                continue
            txt = f"{val:.{prec}f}"
            if abs(val - best[m]) < 1e-9:
                txt = f"**{txt}**" if fmt == "markdown" else rf"\textbf{{{txt}}}"
            cells.append(txt)
        rows.append(cells)
    return (markdown_table(headers, rows) if fmt == "markdown"
            else latex_table(headers, rows, caption, label))


def ablation_table(results: List[Dict], fmt: str = "markdown") -> str:
    has_target = any("target" in r for r in results)
    headers = ["Variant", "Params (M)", "PSNR", "SAM", "ERGAS"]
    if has_target:
        headers += ["PSNR (cross)", "SAM (cross)", "ERGAS (cross)"]
    rows = []
    for r in results:
        cells = [r["variant"], f"{r['params_M']:.2f}",
                 f"{r['source']['psnr']:.3f}", f"{r['source']['sam']:.3f}",
                 f"{r['source']['ergas']:.3f}"]
        if has_target:
            t = r.get("target", {})
            cells += [f"{t.get('psnr', float('nan')):.3f}",
                      f"{t.get('sam', float('nan')):.3f}",
                      f"{t.get('ergas', float('nan')):.3f}"]
        rows.append(cells)
    return (markdown_table(headers, rows) if fmt == "markdown"
            else latex_table(headers, rows, "Component ablation.", "tab:ablation"))


def significance_table(comparisons: List[Dict], metric: str = "psnr",
                       fmt: str = "markdown") -> str:
    headers = ["Baseline", f"ours {metric}", f"baseline {metric}", "delta",
               "Wilcoxon p", "Cohen d", "n"]
    rows = []
    for c in comparisons:
        e = c[metric]
        star = "***" if e["p"] < 0.001 else "**" if e["p"] < 0.01 else \
               "*" if e["p"] < 0.05 else "n.s."
        rows.append([c["name_b"], f"{e['mean_a']:.3f}", f"{e['mean_b']:.3f}",
                     f"{e['delta']:+.3f}", f"{e['p']:.4f} {star}",
                     f"{e['d']:.2f}", int(c["n_scenes"])])
    return (markdown_table(headers, rows) if fmt == "markdown"
            else latex_table(headers, rows, f"Paired significance on {metric}.",
                             f"tab:sig_{metric}"))


# --------------------------------------------------------------------- reports
def environment_report() -> Dict[str, str]:
    """Everything a reviewer needs to reproduce the numbers."""
    import platform
    import sys
    info = {"python": sys.version.split()[0], "platform": platform.platform(),
            "torch": torch.__version__, "numpy": np.__version__,
            "cuda_available": str(torch.cuda.is_available())}
    if torch.cuda.is_available():
        info["gpu"] = torch.cuda.get_device_name(0)
        info["cuda"] = torch.version.cuda or "unknown"
        info["gpu_mem_GB"] = f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f}"
    return info


def save_results(path: str, payload: Dict) -> str:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

    def default(o):
        if isinstance(o, (np.floating, np.integer)):
            return o.item()
        if isinstance(o, np.ndarray):
            return o.tolist()
        return str(o)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=1, default=default)
    return path


def write_report(path: str, title: str, sections: List[Tuple[str, str]]) -> str:
    """Assemble a Markdown report from (heading, body) pairs."""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    parts = [f"# {title}", ""]
    for heading, body in sections:
        parts += [f"## {heading}", "", body, ""]
    text = "\n".join(parts)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    return text

In [ ]:
%%writefile hsifusion/__init__.py
"""hsifusion - shared infrastructure for the HSI-MSI fusion proposals.

One protocol, one metric implementation, one evaluation harness, four
architectures. Each proposal supplies only its model and its own loss terms:

    from hsifusion import BaseConfig, train, evaluate_dataset, FusionLoss

    model, hist = train(cfg, build_model=my_factory, build_loss=my_loss)
    evaluate_dataset(model, cfg.source_root, cfg)

Any model works here as long as it follows the contract
`forward(lr_hsi, msi) -> {"out": tensor}` and carries a `.cfg`. Because the
protocol fields live in `BaseConfig` and the metrics in `metrics.py`, a
difference between two proposals' numbers is attributable to the architecture
rather than to one of them evaluating at a different scale factor - which is
precisely how the ten published baselines in `existing/` became incomparable.
"""

from . import baselines, experiments
from .baselines import BASELINES, evaluate_all_baselines, evaluate_baseline
from .config import BaseConfig
from .data import FusionPatchDataset, SceneCache, estimate_srf
from .degrade import FixedDegradation, blur_downsample, gaussian_kernel2d
from .engine import (clone_state, cosine_lr, evaluate_dataset, evaluate_with_tta,
                     n_params, set_seed, test_time_adapt, tiled_inference, train)
from .io_utils import (available_splits, discover_dataset, find_dataset_roots,
                       find_pairs, infer_channels, load_mat, search_roots,
                       to_chw01)
from .losses import FusionLoss, charbonnier, gradient_loss, mmd_rbf, sam_loss
from .metrics import (evaluate_arrays, metric_ergas, metric_psnr, metric_sam,
                      metric_ssim, ssim_torch)

__version__ = "1.0.0"

__all__ = [
    "BaseConfig", "FusionLoss",
    "train", "evaluate_dataset", "evaluate_with_tta", "test_time_adapt",
    "tiled_inference", "set_seed", "cosine_lr", "clone_state", "n_params",
    "FusionPatchDataset", "SceneCache", "estimate_srf",
    "FixedDegradation", "blur_downsample", "gaussian_kernel2d",
    "discover_dataset", "find_dataset_roots", "find_pairs", "infer_channels",
    "available_splits", "load_mat", "to_chw01", "search_roots",
    "charbonnier", "sam_loss", "gradient_loss", "mmd_rbf",
    "evaluate_arrays", "metric_psnr", "metric_ssim", "metric_sam",
    "metric_ergas", "ssim_torch",
    "BASELINES", "evaluate_baseline", "evaluate_all_baselines",
    "baselines", "experiments", "__version__",
]

## 3. This proposal (`continuumfusion`)

The only part that differs between the four notebooks.

In [ ]:
import os; os.makedirs('continuumfusion', exist_ok=True)

In [ ]:
%%writefile continuumfusion/model.py
"""ContinuumFusion - continuous spectral-spatial representation for HSI-MSI fusion.

WHERE THIS DIFFERS FROM PROPOSALS 1 AND 2
-----------------------------------------
Both earlier proposals produce a fixed-size output grid: the upsampler in
proposal 1 and the decimation operator in proposal 2 are both built for one
scale factor, and changing it means retraining.

ContinuumFusion never represents the image as a grid. It learns a continuous
function

    f(x, y, lambda) -> radiance

conditioned on local features from the observations, and *samples* that function
wherever an output pixel is wanted. The scale factor becomes a query parameter,
not an architectural constant.

WHY THIS IS THE RIGHT GAP TO ATTACK
-----------------------------------
The benchmark in `existing/` is unusable precisely because its ten methods ran
at x4, x8, x16 and x32 and cannot be compared. That is not an accident of
sloppy bookkeeping - it is a property of grid-based architectures, each of
which is welded to the factor it was trained for. A model that handles any
factor from one set of weights makes the comparison well-posed: every method
can be evaluated at every factor.

It also matters physically. A real sensor's resolution ratio is whatever the
optics give you, rarely a power of two.

THE ARCHITECTURE
----------------
1. Encoder: LR-HSI and MSI are encoded to a feature map on the *LR* grid,
   because that is where the hyperspectral information actually lives.
2. Continuous decoder: for an output coordinate (x, y), gather the four nearest
   LR feature vectors, and for each, feed [feature, delta_coord, cell_size] to
   an MLP that predicts the radiance. Blend the four by area weights. This is
   local implicit image function decoding, made spectral.
3. Spectral coordinate: the band index enters as a coordinate too, through a
   learned per-band embedding, so the same MLP produces all bands and the
   representation is continuous along wavelength as well. That is what allows
   querying bands the sensor never sampled.
4. MSI detail is injected at the queried coordinate by bilinear sampling of the
   HR MSI feature map, so high-frequency spatial information is read at full
   resolution rather than upsampled from the LR grid.

WHY IT SHOULD TRANSFER
----------------------
The decoder sees relative coordinates and cell sizes, never absolute image
size, so it is scale-agnostic by construction. Training with randomised scale
factors turns the resolution ratio into just another nuisance variable the
model has learned to be robust to - the same argument as randomising the blur,
applied to geometry.
"""

from __future__ import annotations

import math
from typing import Dict, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F


# --------------------------------------------------------------------- encoder
class ResBlock(nn.Module):
    def __init__(self, ch: int):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(ch, ch, 3, 1, 1), nn.ReLU(inplace=True),
            nn.Conv2d(ch, ch, 3, 1, 1))

    def forward(self, x):
        return x + self.body(x)


class Encoder(nn.Module):
    """Features on the LR grid, fused with MSI context pooled to that grid.

    The latent lives on the LR grid on purpose: the spectra are only measured
    there, so that is where spectral reasoning belongs. Spatial detail is added
    later, at query time, from the full-resolution MSI.
    """

    def __init__(self, bands: int, msi_bands: int, width: int, depth: int):
        super().__init__()
        self.hsi_stem = nn.Conv2d(bands, width, 3, 1, 1)
        self.msi_stem = nn.Conv2d(msi_bands, width, 3, 1, 1)
        self.fuse = nn.Conv2d(width * 2, width, 1)
        self.body = nn.Sequential(*[ResBlock(width) for _ in range(depth)])

    def forward(self, lr_hsi: torch.Tensor, msi: torch.Tensor) -> torch.Tensor:
        h = self.hsi_stem(lr_hsi)
        m = self.msi_stem(msi)
        m_lr = F.adaptive_avg_pool2d(m, lr_hsi.shape[-2:])
        z = self.fuse(torch.cat([h, m_lr], dim=1))
        return self.body(z)


class DetailEncoder(nn.Module):
    """High-resolution MSI features, sampled at query coordinates."""

    def __init__(self, msi_bands: int, width: int):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(msi_bands, width, 3, 1, 1), nn.ReLU(inplace=True),
            ResBlock(width), nn.Conv2d(width, width, 3, 1, 1))

    def forward(self, msi: torch.Tensor) -> torch.Tensor:
        return self.body(msi)


# ------------------------------------------------------------ continuous decoder
class SpectralCoordMLP(nn.Module):
    """Decodes (latent, relative coordinate, cell size, band embedding) -> radiance.

    The band index is a *coordinate*, not an output channel. One MLP therefore
    serves every band, the parameter count does not grow with band count, and
    the representation is continuous along wavelength - so a band the sensor
    never sampled can be queried by interpolating its embedding.
    """

    def __init__(self, latent: int, detail: int, bands: int, width: int,
                 depth: int, band_dim: int = 24):
        super().__init__()
        self.band_emb = nn.Parameter(torch.randn(bands, band_dim) * 0.02)
        in_dim = latent + detail + 2 + 2 + band_dim   # +delta_xy +cell_hw +band
        layers, d = [], in_dim
        for _ in range(depth):
            layers += [nn.Linear(d, width), nn.ReLU(inplace=True)]
            d = width
        layers += [nn.Linear(d, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, latent: torch.Tensor, detail: torch.Tensor,
                delta: torch.Tensor, cell: torch.Tensor) -> torch.Tensor:
        """latent [N,Cl] detail [N,Cd] delta [N,2] cell [N,2] -> [N,bands]."""
        n, bands = latent.shape[0], self.band_emb.shape[0]
        base = torch.cat([latent, detail, delta, cell], dim=-1)     # [N,D]
        base = base[:, None, :].expand(n, bands, base.shape[-1])
        emb = self.band_emb[None].expand(n, bands, -1).to(base.dtype)
        return self.net(torch.cat([base, emb], dim=-1)).squeeze(-1)  # [N,bands]


def make_coord(shape: Tuple[int, int], device, flatten: bool = True
               ) -> torch.Tensor:
    """Pixel-centre coordinates in [-1, 1]. Centres, not corners: using corners
    biases every interpolation by half a pixel."""
    coords = []
    for n in shape:
        r = 1.0 / n
        coords.append(torch.arange(n, device=device).float() * 2 * r - 1 + r)
    grid = torch.stack(torch.meshgrid(*coords, indexing="ij"), dim=-1)
    return grid.reshape(-1, 2) if flatten else grid


class ContinuumFusion(nn.Module):
    """HSI-MSI fusion as a continuous field, queried at arbitrary resolution."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.encoder = Encoder(cfg.bands, cfg.msi_bands, cfg.width, cfg.enc_depth)
        self.detail = DetailEncoder(cfg.msi_bands, cfg.detail_width)
        # feature unfolding: a 3x3 neighbourhood gives the decoder local context
        latent = cfg.width * 9 if cfg.unfold else cfg.width
        self.decoder = SpectralCoordMLP(latent, cfg.detail_width, cfg.bands,
                                        cfg.mlp_width, cfg.mlp_depth,
                                        cfg.band_dim)
        self.unfold = cfg.unfold

    # ------------------------------------------------------------------ query
    def query(self, feat: torch.Tensor, detail: torch.Tensor,
              coord: torch.Tensor, cell: torch.Tensor) -> torch.Tensor:
        """Sample the continuous field at `coord`.

        Local ensemble: each output point is decoded from its four neighbouring
        latent cells and blended by area weights. Decoding from the single
        nearest cell leaves visible blocking at the cell boundaries.
        """
        b, c, h, w = feat.shape
        if self.unfold:
            feat = F.unfold(feat, 3, padding=1).view(b, c * 9, h, w)

        feat_coord = make_coord((h, w), feat.device, flatten=False)
        feat_coord = feat_coord.permute(2, 0, 1)[None].expand(b, 2, h, w)

        rx, ry = 1.0 / h, 1.0 / w
        preds, areas = [], []
        for dx in (-1, 1):
            for dy in (-1, 1):
                c_ = coord.clone()
                c_[..., 0] += dx * rx + 1e-6
                c_[..., 1] += dy * ry + 1e-6
                c_.clamp_(-1 + 1e-6, 1 - 1e-6)
                grid = c_.flip(-1)[:, :, None, :]                  # [B,N,1,2]
                q_feat = F.grid_sample(feat, grid, mode="nearest",
                                       align_corners=False)[:, :, :, 0]
                q_coord = F.grid_sample(feat_coord, grid, mode="nearest",
                                        align_corners=False)[:, :, :, 0]
                q_feat = q_feat.permute(0, 2, 1)                   # [B,N,C]
                q_coord = q_coord.permute(0, 2, 1)                 # [B,N,2]
                rel = coord - q_coord
                rel[..., 0] *= h
                rel[..., 1] *= w
                # detail read at full resolution, not upsampled from the LR grid
                q_det = F.grid_sample(detail, coord.flip(-1)[:, :, None, :],
                                      mode="bilinear", align_corners=False
                                      )[:, :, :, 0].permute(0, 2, 1)
                n = coord.shape[1]
                pred = self.decoder(q_feat.reshape(b * n, -1),
                                    q_det.reshape(b * n, -1),
                                    rel.reshape(b * n, 2),
                                    cell.reshape(b * n, 2))
                preds.append(pred.view(b, n, -1))
                areas.append(torch.abs(rel[..., 0] * rel[..., 1]) + 1e-9)

        # diagonal swap: weight each corner by the area of the opposite rectangle
        areas = [areas[3], areas[2], areas[1], areas[0]]
        tot = sum(areas)
        out = sum(p * (a / tot)[..., None] for p, a in zip(preds, areas))
        return out                                                # [B,N,bands]

    def forward(self, lr_hsi: torch.Tensor, msi: torch.Tensor,
                out_hw: Optional[Tuple[int, int]] = None
                ) -> Dict[str, torch.Tensor]:
        """`out_hw` defaults to the MSI grid, but any resolution can be asked
        for - that is the whole point of the representation."""
        b = lr_hsi.shape[0]
        if out_hw is None:
            out_hw = (msi.shape[-2], msi.shape[-1])
        h, w = out_hw

        feat = self.encoder(lr_hsi, msi)
        detail = self.detail(msi)

        coord = make_coord((h, w), lr_hsi.device)[None].expand(b, h * w, 2)
        cell = torch.ones_like(coord)
        cell[..., 0] *= 2.0 / h
        cell[..., 1] *= 2.0 / w

        # the residual over a cheap interpolation keeps the MLP predicting a
        # correction rather than the full radiance, which trains far faster
        base = F.interpolate(lr_hsi, size=out_hw, mode="bicubic",
                             align_corners=False)
        pred = self.query(feat, detail, coord, cell)               # [B,N,bands]
        pred = pred.permute(0, 2, 1).reshape(b, -1, h, w)
        out = (base + pred).clamp(0, 1)
        return {"out": out, "residual": pred, "feat": feat.mean(dim=(2, 3))}

    def features(self, lr_hsi: torch.Tensor, msi: torch.Tensor) -> torch.Tensor:
        return self.encoder(lr_hsi, msi).mean(dim=(2, 3))


def build_model(cfg) -> ContinuumFusion:
    return ContinuumFusion(cfg)

In [ ]:
%%writefile continuumfusion/__init__.py
"""ContinuumFusion - arbitrary-scale HSI-MSI fusion by implicit representation.

Proposal 3. Proposals 1 and 2 are welded to one scale factor; this one treats
resolution as a query parameter, so a single set of weights serves x4 through
x32 - which is what makes a like-for-like comparison across factors possible
at all.

    import continuumfusion as C
    cfg = C.Config().resolve()
    model, hist = C.train(cfg)
    out = model(lr, msi, out_hw=(1024, 1024))["out"]   # any resolution
"""

from dataclasses import dataclass, field
from typing import Tuple

from hsifusion import BaseConfig, FusionLoss
from hsifusion import engine as _engine

from .model import (ContinuumFusion, DetailEncoder, Encoder, SpectralCoordMLP,
                    build_model, make_coord)

__version__ = "1.0.0"


@dataclass
class Config(BaseConfig):
    name: str = "continuumfusion"
    width: int = 64            # LR latent width
    enc_depth: int = 4
    detail_width: int = 32     # HR MSI detail width
    mlp_width: int = 128
    mlp_depth: int = 4
    band_dim: int = 24         # spectral coordinate embedding
    unfold: bool = True        # 3x3 latent neighbourhood as decoder context
    # scale factors sampled during training. Randomising the factor is what
    # makes the resolution ratio a nuisance variable rather than a constant.
    train_scales: Tuple[int, ...] = (2, 3, 4, 6, 8)
    out_dir: str = "./continuum_out"


def build_loss(cfg, srf):
    import torch
    return FusionLoss(cfg, torch.from_numpy(srf))


def train(cfg, device: str = "cuda", **kw):
    return _engine.train(cfg, build_model=build_model, build_loss=build_loss,
                         device=device, **kw)


def evaluate_scales(model, root, cfg, scales=(4, 8, 16, 32), device="cuda",
                    limit=None, verbose=True):
    """Score one trained model at several scale factors.

    This is the experiment the grid-based proposals cannot run without
    retraining, and the one that makes the x4/x8/x16/x32 confusion in
    `existing/` addressable.
    """
    import copy

    from hsifusion.engine import evaluate_dataset

    rows = []
    for s in scales:
        c = copy.deepcopy(cfg)
        c.scale = s
        c.patch = max(c.patch, s * 8)
        if verbose:
            print(f"\n--- scale x{s} ---")
        m = evaluate_dataset(model, root, c, "Test", device, limit=limit,
                             verbose=verbose)
        rows.append({"scale": s, **m})
    return rows


__all__ = ["Config", "ContinuumFusion", "build_model", "build_loss", "train",
           "evaluate_scales", "Encoder", "DetailEncoder", "SpectralCoordMLP",
           "make_coord", "__version__"]

In [ ]:
import sys
for m in [k for k in list(sys.modules) if k.startswith(('hsifusion', 'continuumfusion'))]:
    del sys.modules[m]
sys.path.insert(0, os.getcwd())
import hsifusion
import continuumfusion
print('hsifusion', hsifusion.__version__, '|', 'continuumfusion', continuumfusion.__version__)

## 4. Configuration

Dataset roots and band counts are discovered from the filesystem - nothing is
hardcoded. Attach the CAVE and Harvard datasets and this finds them.

In [ ]:
QUICK = False        # True -> a few minutes, to validate the pipeline first
DEVICE = 'cuda' if (torch.cuda.is_available() and GPU_OK) else 'cpu'
if DEVICE == 'cpu' and torch.cuda.is_available():
    raise RuntimeError('Unsupported GPU - switch the accelerator to "GPU T4 x2".')

cfg = continuumfusion.Config(
    scale=4,          # identical across all four proposals
    patch=64,
    out_dir=os.path.join(os.getcwd(), 'continuumfusion_out'),
)
if QUICK:
    cfg.iters, cfg.batch, cfg.val_every, cfg.log_every = 300, 8, 150, 50

cfg.resolve()
print(json.dumps({k: v for k, v in cfg.to_dict().items()
                  if k in ('name','source_root','target_root','bands',
                           'msi_bands','scale','patch','batch','iters')}, indent=1))

## 5. Training

Trained on the source domain only. The target dataset is never used with labels.

In [ ]:
t0 = time.time()
model, history = continuumfusion.train(cfg, device=DEVICE)
print(f'\ntrained in {(time.time() - t0) / 60:.1f} min')

srf = torch.load(os.path.join(cfg.out_dir, f'{cfg.name}_final.pth'),
                 map_location='cpu', weights_only=False)['srf']

In [ ]:
import matplotlib.pyplot as plt
if history and history['iter']:
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].plot(history['iter'], history['loss'])
    ax[0].set_xlabel('iteration'); ax[0].set_ylabel('loss'); ax[0].grid(alpha=.3)
    if history['val']:
        it = [v['iter'] for v in history['val']]
        ax[1].plot(it, [v['psnr'] for v in history['val']], marker='o')
        ax[1].set_xlabel('iteration'); ax[1].set_ylabel('PSNR (dB)')
        ax[1].grid(alpha=.3)
        axb = ax[1].twinx()
        axb.plot(it, [v['sam'] for v in history['val']], marker='s', color='tab:red')
        axb.set_ylabel('SAM (deg)')
    plt.tight_layout(); plt.savefig('training_curves.png', dpi=140); plt.show()

## 6. In-domain and cross-domain evaluation

Full scenes via Hann-weighted overlapping tiles, scored by the shared metric
module with a fixed `data_range=1.0` - never the per-image maximum, which is
what inflated several numbers in the original benchmark.

In [ ]:
print('=== in-domain (source Test) ===')
src_mean, src_rows = hsifusion.evaluate_dataset(
    model, cfg.source_root, cfg, 'Test', DEVICE, return_rows=True)

tgt_mean = tgt_rows = tta_mean = tta_rows = None
if cfg.target_root:
    print('\n=== zero-shot cross-domain (target Test) ===')
    tgt_mean, tgt_rows = hsifusion.evaluate_dataset(
        model, cfg.target_root, cfg, 'Test', DEVICE, return_rows=True)

    print('\n=== cross-domain + physics-only test-time adaptation ===')
    crit = continuumfusion.build_loss(cfg, srf).to(DEVICE)
    tta_mean, tta_rows = hsifusion.evaluate_with_tta(
        model, cfg.target_root, cfg, crit, 'Test', DEVICE, steps=30)

## 7. Same-protocol baselines and paired significance

These classical methods need no checkpoints, so they run through the identical
pipeline. They are the only rows strictly comparable to ours, and they set the
floor any learned method must clear.

In [ ]:
print('=== classical baselines, source domain ===')
base_src = hsifusion.evaluate_all_baselines(cfg.source_root, cfg, srf, 'Test',
                                            DEVICE, verbose=False)
for name, r in base_src.items():
    m = r['mean']
    print(f'  {name:<14} PSNR={m["psnr"]:7.3f}  SSIM={m["ssim"]:.4f}  '
          f'SAM={m["sam"]:6.3f}  ERGAS={m["ergas"]:8.3f}')

base_tgt = None
if cfg.target_root:
    print('\n=== classical baselines, target domain ===')
    base_tgt = hsifusion.evaluate_all_baselines(cfg.target_root, cfg, srf,
                                                'Test', DEVICE, verbose=False)
    for name, r in base_tgt.items():
        m = r['mean']
        print(f'  {name:<14} PSNR={m["psnr"]:7.3f}  SSIM={m["ssim"]:.4f}  '
              f'SAM={m["sam"]:6.3f}  ERGAS={m["ergas"]:8.3f}')

In [ ]:
from hsifusion.experiments import (comparison_table, compare_methods,
                                   significance_table, summarise_rows)

entries = {k: v['mean'] for k, v in base_src.items()}
entries[f'{cfg.name} (in-domain)'] = src_mean
print('=== SOURCE DOMAIN, same protocol ===')
print(comparison_table(entries))

if tgt_mean and base_tgt:
    t = {k: v['mean'] for k, v in base_tgt.items()}
    t[f'{cfg.name} (zero-shot)'] = tgt_mean
    if tta_mean:
        t[f'{cfg.name} (+TTA)'] = tta_mean
    print('\n=== TARGET DOMAIN, same protocol ===')
    print(comparison_table(t))

print('\n=== paired significance vs same-protocol baselines (SAM) ===')
cmps = [compare_methods(src_rows, r['rows'], cfg.name, name)
        for name, r in base_src.items()]
print(significance_table(cmps, metric='sam'))

## 8. The claim: one model, every scale factor

This is the experiment the other three proposals cannot run without retraining,
and the one that makes the x4/x8/x16/x32 confusion in `existing/` addressable.

In [ ]:
scale_rows = continuumfusion.evaluate_scales(
    model, cfg.source_root, cfg, scales=(4, 8, 16, 32), device=DEVICE,
    limit=4, verbose=False)
print(f'{"scale":>6} {"PSNR":>8} {"SSIM":>8} {"SAM":>8} {"ERGAS":>9}')
for r in scale_rows:
    print(f'x{r["scale"]:<5} {r["psnr"]:>8.3f} {r["ssim"]:>8.4f} '
          f'{r["sam"]:>8.3f} {r["ergas"]:>9.3f}')
print('\nall rows from ONE set of weights - no retraining between them')

print('\n--- non-integer and non-square queries ---')
import torch
pairs = hsifusion.find_pairs(cfg.source_root, 'Test')
cache = hsifusion.SceneCache(cfg.bands, cfg.msi_bands, limit=1)
hsi, rgb = cache.get(*pairs[0])
side = (min(hsi.shape[1], 256) // 32) * 32
gt = torch.from_numpy(hsi[:, :side, :side].astype('float32'))[None].to(DEVICE)
msi_q = torch.from_numpy(rgb[:, :side, :side].astype('float32'))[None].to(DEVICE)
lr_q = hsifusion.FixedDegradation(cfg.scale, cfg.blur_ksize,
                                  cfg.eval_sigma).to(DEVICE)(gt)
model.eval()
with torch.no_grad():
    for hw in [(side, side), (side // 2, side), (int(side * 1.5), side)]:
        o = model(lr_q, msi_q, out_hw=hw)['out']
        print(f'  query {str(hw):16s} -> {tuple(o.shape[-2:])}')

## 9. Cost and saved results

In [ ]:
try:
    prof = hsifusion.experiments.profile_model(model, cfg, device=DEVICE, hr=512)
    for k, v in prof.items():
        print(f'  {k:14s} {v:.3f}' if isinstance(v, float) else f'  {k:14s} {v}')
except Exception as exc:
    prof = {'note': f'profile skipped: {exc}'}
    print(prof['note'])

In [ ]:
from hsifusion.experiments import environment_report, save_results, write_report

payload = {
    'proposal': cfg.name,
    'config': cfg.to_dict(),
    'environment': environment_report(),
    'efficiency': prof,
    'source': {'mean': src_mean, 'rows': src_rows,
               'summary': summarise_rows(src_rows)},
    'baselines_same_protocol': {
        'source': {k: v['mean'] for k, v in base_src.items()},
        'target': ({k: v['mean'] for k, v in base_tgt.items()} if base_tgt else None)},
    'significance_vs_same_protocol': cmps,
}
if tgt_rows:
    payload['target_zeroshot'] = {'mean': tgt_mean, 'rows': tgt_rows,
                                  'summary': summarise_rows(tgt_rows)}
if tta_rows:
    payload['target_tta'] = {'mean': tta_mean, 'rows': tta_rows,
                             'summary': summarise_rows(tta_rows)}

save_results(f'{cfg.name}_results.json', payload)
sections = [('Environment', '\n'.join(f'- **{k}**: {v}'
             for k, v in environment_report().items())),
            ('Results', comparison_table(entries))]
write_report(f'{cfg.name}_RESULTS.md', f'{cfg.name} run report', sections)
print('written:', [f for f in os.listdir('.') if f.endswith(('.json', '.md', '.png'))])